In [ ]:
import pandas as pd

df = pd.read_csv("twcs.csv", engine="python", on_bad_lines="warn")

In [ ]:
print("shape:", df.shape)

shape: (943193, 7)


In [ ]:
print(df.dtypes)
print(df["inbound"].value_counts())

tweet_id                     int64
author_id                   object
inbound                     object
created_at                  object
text                        object
response_tweet_id           object
in_response_to_tweet_id    float64
dtype: object
inbound
True     517288
False    425904
Name: count, dtype: int64


In [ ]:
print(df["inbound"].dtype)
print(df["inbound"].unique()[:20])

object
[False True nan]


In [ ]:
print(df["inbound"].isna().sum())  # see how many rows are affected

1


In [ ]:
df = df.dropna(subset=["inbound"])
df["inbound"] = df["inbound"].astype(bool)

In [ ]:
# Brands = authors that are never inbound (i.e. always the support account)
brand_ids = df.loc[~df["inbound"], "author_id"].value_counts()
print(brand_ids.head(30))

author_id
AmazonHelp         72093
AppleSupport       29581
Uber_Support       19269
Delta              14539
SpotifyCares       12929
AmericanAir        12588
British_Airways    10156
comcastcares        9777
Tesco               9711
TMobileHelp         9556
VirginTrains        9521
XboxSupport         9354
SouthwestAir        9047
hulu_support        8582
AskPlayStation      7731
Safaricom_Care      6516
ChipotleTweets      6386
VerizonSupport      6351
sprintcare          6292
Ask_Spectrum        6120
sainsburys          6096
idea_cares          5866
GWRHelp             5810
ATVIAssist          5728
UPSHelp             5268
MicrosoftHelps      5009
O2                  4981
BofA_Help           4679
AskAmex             4252
AskeBay             3785
Name: count, dtype: int64


In [ ]:
# distinct customer threads per brand = count of unique in_response_to_tweet_id
# that brand tweets reply to (approx proxy for "how many separate conversations")
brand_tweets = df[~df["inbound"]]
threads_per_brand = brand_tweets.groupby("author_id")["in_response_to_tweet_id"].nunique().sort_values(ascending=False)
print(threads_per_brand.head(30))

author_id
AmazonHelp         65540
AppleSupport       29503
Uber_Support       18993
Delta              12953
SpotifyCares       12616
AmericanAir        12497
TMobileHelp         9444
comcastcares        9207
VirginTrains        9140
British_Airways     8953
SouthwestAir        8868
hulu_support        8467
XboxSupport         8202
AskPlayStation      7551
Tesco               7362
ChipotleTweets      6310
VerizonSupport      6237
Ask_Spectrum        5946
sprintcare          5808
Safaricom_Care      5722
GWRHelp             5633
sainsburys          5623
ATVIAssist          5487
UPSHelp             5204
idea_cares          5132
O2                  4888
MicrosoftHelps      4167
BofA_Help           3655
McDonalds           3633
ArgosHelpers        3477
Name: in_response_to_tweet_id, dtype: int64


In [ ]:
for brand in ["AppleSupport", "AmazonHelp", "Delta", "Uber_Support", "SpotifyCares"]:  # adjust to your top candidates
    print(f"\n=== {brand} ===")
    sample = df[df["author_id"] == brand].sample(15, random_state=42)
    for t in sample["text"]:
        print("-", t[:180])


=== AppleSupport ===
- @312019 Please restart your Apple Watch and test for us.  This link explains how to do this: https://t.co/gZrStAoDfj
- @327762 Let's take things to DM for the next steps. https://t.co/GDrqU22YpT
- @283937 We want to help. Which device are you using iBooks with? Tell us in DM. https://t.co/GDrqU22YpT
- @314068 We can happily help. Have you tried a different charger or wall outlet by chance? If not, give that a try for us.
- @358318 We're glad to hear that. Let us know if you have any further questions. Enjoy!
- @312886 Let's get you back to enjoying your iPhone. What seems to be going on with the iPhone? Also, which version of iOS do you have?
- @262112 We're sorry to hear you're having issues with your iPhone. That is not the experience we want for our customers. Send us a DM with details of what is going on. We'll follow
- @310424 Thanks for reaching out! We can certainly help you. Which macOS are you running currently? https://t.co/GDrqU22YpT
- @253831 Thank y

In [ ]:
brand = "AppleSupport"

# get every tweet authored by the brand, plus everything connected via reply chains
brand_tweets = df[df["author_id"] == brand]
relevant_ids = set(brand_tweets["tweet_id"]) | set(brand_tweets["in_response_to_tweet_id"].dropna())

# expand outward a couple hops to catch full conversations (parents-of-parents, children-of-children)
for _ in range(3):
    linked = df[
        df["tweet_id"].isin(relevant_ids)
        | df["in_response_to_tweet_id"].isin(relevant_ids)
    ]
    new_ids = set(linked["tweet_id"]) | set(linked["in_response_to_tweet_id"].dropna())
    if new_ids == relevant_ids:
        break
    relevant_ids = new_ids

brand_df = df[df["tweet_id"].isin(relevant_ids)].copy()
brand_df.to_csv(f"{brand}_filtered.csv", index=False)
print(brand_df.shape)

(65706, 7)


In [ ]:
brand_df["created_at_parsed"] = pd.to_datetime(brand_df["created_at"], errors="coerce", utc=True)
print(brand_df["created_at_parsed"].min(), "to", brand_df["created_at_parsed"].max())
print(brand_df["created_at_parsed"].dt.to_period("M").value_counts().sort_index())

/tmp/ipykernel_724/2479916714.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  brand_df["created_at_parsed"] = pd.to_datetime(brand_df["created_at"], errors="coerce", utc=True)


2016-03-04 01:19:41+00:00 to 2017-12-03 23:12:28+00:00
created_at_parsed
2016-03        2
2016-04        1
2016-07        1
2017-01        8
2017-02        7
2017-03        6
2017-04       11
2017-05        5
2017-07        2
2017-08        7
2017-09      249
2017-10    38521
2017-11    16524
2017-12    10362
Freq: M, Name: count, dtype: int64


/tmp/ipykernel_724/2479916714.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  print(brand_df["created_at_parsed"].dt.to_period("M").value_counts().sort_index())


In [ ]:
print('inbound counts:', df['inbound'].value_counts().to_dict())
print()
print('n unique author_ids:', df['author_id'].nunique())
print()
print('is AppleSupport always non-inbound?', df[df.author_id=='AppleSupport']['inbound'].unique())
print()
print('n AppleSupport rows:', (df.author_id=='AppleSupport').sum())
print()
# date range


inbound counts: {True: 517288, False: 425904}

n unique author_ids: 248229

is AppleSupport always non-inbound? [False]

n AppleSupport rows: 29581



In [ ]:
import sys
from threads import load_raw, build_brand_threads
import time

t0 = time.time()
df = load_raw('AppleSupport_filtered.csv')
print('loaded', df.shape, f'in {time.time()-t0:.1f}s')

t0 = time.time()
threads = build_brand_threads(df, 'AppleSupport')
print(f'built {len(threads)} threads in {time.time()-t0:.1f}s')

loaded (65706, 7) in 0.7s
built 20311 threads in 0.4s


In [ ]:
import sys
from threads import load_raw, build_brand_threads
import pandas as pd
from collections import Counter

df = load_raw('AppleSupport_filtered.csv')
threads = build_brand_threads(df, 'AppleSupport')

lengths = [len(t.turns) for t in threads]
print('thread length distribution:')
print(pd.Series(lengths).describe())
print()
print('length value counts (top 10):')
print(pd.Series(lengths).value_counts().sort_index().head(10))
print()

has_reply = sum(1 for t in threads if t.has_brand_reply)
print(f'threads with a brand reply: {has_reply} / {len(threads)} ({100*has_reply/len(threads):.1f}%)')
print()

# how many threads have >2 turns (i.e., actual back-and-forth, not just ask+ack)
multi_turn = sum(1 for t in threads if len(t.turns) > 2)
print(f'threads with >2 turns (real back-and-forth): {multi_turn} ({100*multi_turn/len(threads):.1f}%)')
print()

# does thread end on agent turn (closed) or customer turn (still open / no closure)
ends_on_agent = sum(1 for t in threads if t.turns and not t.turns[-1].is_inbound)
print(f'threads ending on an agent turn: {ends_on_agent} ({100*ends_on_agent/len(threads):.1f}%)')

thread length distribution:
count    20311.000000
mean         3.015213
std          1.540849
min          2.000000
25%          2.000000
50%          2.000000
75%          4.000000
max         26.000000
dtype: float64

length value counts (top 10):
2     11802
3      2238
4      3683
5       853
6      1025
7       250
8       288
9        65
10       73
11       21
Name: count, dtype: int64

threads with a brand reply: 20311 / 20311 (100.0%)

threads with >2 turns (real back-and-forth): 8509 (41.9%)

threads ending on an agent turn: 17750 (87.4%)


In [ ]:
import sys
from threads import load_raw, build_brand_threads
import pandas as pd

df = load_raw('AppleSupport_filtered.csv')
threads = build_brand_threads(df, 'AppleSupport')

# build a dataframe of thread-opening customer messages with their month
rows = []
for t in threads:
    if not t.turns:
        continue
    first = t.turns[0]
    if not first.is_inbound:
        continue  # skip threads that don't open with a customer message (rare edge case)
    rows.append({
        'thread_id': t.thread_id,
        'opening_text': first.text,
        'created_at': first.created_at,
        'n_turns': len(t.turns),
    })

opens = pd.DataFrame(rows)
opens['dt'] = pd.to_datetime(opens['created_at'], errors='coerce', utc=True)
opens['month'] = opens['dt'].dt.to_period('M')
print('total threads opening with a customer message:', len(opens))
print()
print(opens['month'].value_counts().sort_index())
opens.to_csv('thread_openers.csv', index=False)

/tmp/ipykernel_724/2816020126.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  opens['dt'] = pd.to_datetime(opens['created_at'], errors='coerce', utc=True)
/tmp/ipykernel_724/2816020126.py:25: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  opens['month'] = opens['dt'].dt.to_period('M')


total threads opening with a customer message: 20213

month
2016-03        1
2016-07        1
2017-01        2
2017-02        1
2017-04        2
2017-05        2
2017-07        1
2017-08        1
2017-09       95
2017-10    11658
2017-11     5109
2017-12     3340
Freq: M, Name: count, dtype: int64


In [ ]:
print('saved', len(opens), 'rows')

saved 20213 rows


In [ ]:
opens = pd.read_csv('thread_openers.csv')

# restrict to the 4 real months with meaningful volume; treat the handful of
# stray earlier rows as noise (documented) and exclude from taxonomy sample
opens = opens[opens['month'].isin(['2017-09','2017-10','2017-11','2017-12'])].copy()

# stratified sample: 40 per month (Sept has only 95 total, so cap there)
samples = []
for month, group in opens.groupby('month'):
    n = min(40, len(group))
    samples.append(group.sample(n, random_state=42))
sample_df = pd.concat(samples).sort_values('month')
print('sample size:', len(sample_df))
print(sample_df['month'].value_counts())
sample_df.to_csv('taxonomy_induction_sample.csv', index=False)

sample size: 160
month
2017-09    40
2017-10    40
2017-11    40
2017-12    40
Name: count, dtype: int64


In [ ]:
df = pd.read_csv("taxonomy_induction_sample.csv")

for _, row in df.head(10).iterrows():
    print(f"[{row['month']}] {row['opening_text']}")

[2017-09] .@312398 Is the 2015 i30's iPod mode incompatible with iOS 11? It keeps displaying "Communication Error".
[2017-09] A new generation of iPhone. Now available.
[2017-09] My iPhone 7 doesnthave an "auto call answering" option but I sick of it auto answering after few rings. How do I stop it?😡😡💚💚 @AppleSupport
[2017-09] @applesupport I’m missing several thousand photos after upgrading to iOS 11. Years worth of content. I sure hope you have a solution.
[2017-09] .@AppleSupport iOS11 - when playing podcasts, the lock screen doesn’t show correct play times until paused. Could you log a fault? Thanks.
[2017-09] This is what my lock screen looks like, @115858 PLEASE address the bugs. I have like 4 other glitches that I can recreate with the new update 😣 https://t.co/NzAfYNzZpt
[2017-09] Hey @AppleSupport ever since I updated to iOS 11, my iPhone 6S keeps just kinda shutting down randomly and turning back on immediately. It’s rather annoying. Can you make it stop doing that please
[20

In [ ]:
opens = pd.read_csv('thread_openers.csv')
opens = opens[opens['month'].isin(['2017-09','2017-10','2017-11','2017-12'])].copy()
print('pool size across 4 real months:', len(opens))
print(opens['month'].value_counts())

pool size across 4 real months: 20202
month
2017-10    11658
2017-11     5109
2017-12     3340
2017-09       95
Name: count, dtype: int64


In [ ]:
opens = opens[opens['month'].isin(['2017-09','2017-10','2017-11','2017-12'])].copy()

# oversample pool: pull more than 250 since some will be NOT_A_REQUEST/junk
# and we'll want to cap category 1 later, so we need surplus in other categories.
# sqrt-weighted by month so Sept/Dec aren't drowned by Oct, but Oct still dominates.
samples = []
per_month_target = {'2017-09': 30, '2017-10': 200, '2017-11': 150, '2017-12': 120}
for month, n in per_month_target.items():
    pool = opens[opens['month'] == month]
    n = min(n, len(pool))
    samples.append(pool.sample(n, random_state=7))

candidate_pool = pd.concat(samples).sample(frac=1, random_state=7).reset_index(drop=True)
candidate_pool['label_intent'] = ''
candidate_pool['label_escalate'] = ''
candidate_pool['label_reason'] = ''
candidate_pool.to_csv('golden_set_candidates.csv', index=False)
print('candidate pool size:', len(candidate_pool))
print(candidate_pool.columns.tolist())

candidate pool size: 500
['thread_id', 'opening_text', 'created_at', 'n_turns', 'dt', 'month', 'label_intent', 'label_escalate', 'label_reason']
